# 03 Global Nuclear EDA

Operating fleet: capacity, technology mix, geography, fleet age, and historical buildout.

> Run the pipeline first: `python run_pipeline.py`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed = ROOT / 'data' / 'processed'
predictions = ROOT / 'outputs' / 'predictions'

reactors = pd.read_csv(processed / 'reactors_master.csv', parse_dates=['commercial_operation_date'])
countries = pd.read_csv(processed / 'country_nuclear_profile.csv')
print(f"Reactor records: {len(reactors)}  |  Countries: {reactors.country.nunique()}")
reactors.groupby('status_group').agg(units=('reactor_id','count'), capacity_gwe=('capacity_mwe', lambda x: round(x.sum()/1000,1))).sort_values('capacity_gwe', ascending=False)

## Operating capacity by country

In [ ]:
operating = reactors[reactors.status_group == 'Operating']
cap = operating.groupby('country')['capacity_mwe'].sum().div(1000).sort_values(ascending=False).head(12).reset_index()
cap.columns = ['country','capacity_gwe']

fig, ax = plt.subplots(figsize=(12,5))
sns.barplot(data=cap, x='country', y='capacity_gwe', ax=ax)
ax.set_title('Operating Nuclear Capacity by Country (GWe)', fontsize=14)
ax.set_xlabel(''); ax.set_ylabel('GWe')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

## Technology mix — operating fleet

In [ ]:
tech = operating.groupby('reactor_type_standardized')['capacity_mwe'].sum().sort_values(ascending=False).reset_index()
px.pie(tech, names='reactor_type_standardized', values='capacity_mwe',
       title='Operating Fleet – Reactor Type Mix (by capacity)', template='plotly_white').show()

**LWR dominance reflects 70 years of commercial deployment.** PWR/VVER/BWR/AP1000/EPR together account for the vast majority of operating capacity. This is why advanced technologies score lower on the maturity index — a deliberate design choice in the scoring model.

## Fleet age distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

med = operating['age_years'].median()
sns.histplot(operating['age_years'].dropna(), bins=20, ax=axes[0], color='steelblue')
axes[0].set_title('Operating Fleet Age Distribution')
axes[0].set_xlabel('Age (years)')
axes[0].axvline(med, color='red', linestyle='--', label=f'Median: {med:.0f} yr')
axes[0].legend()

age_c = operating.groupby('country')['age_years'].mean().sort_values(ascending=False).head(12).reset_index()
sns.barplot(data=age_c, x='country', y='age_years', ax=axes[1], palette='flare')
axes[1].set_title('Average Fleet Age – top 12 oldest countries')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout(); plt.show()
print(f"Fleet median age: {med:.0f} years")

**Aging fleet = replacement pressure.** Many operating reactors are 30–45 years old. Countries with the oldest fleets face a capacity cliff unless they build or extend — a structural demand driver for new projects.

## Nuclear share vs GDP per capita

In [ ]:
c = countries.dropna(subset=['gdp_per_capita','nuclear_share_percent']).query('operating_capacity_mwe > 0')
px.scatter(c, x='gdp_per_capita', y='nuclear_share_percent',
           size='operating_capacity_mwe', color='region', hover_name='country', log_x=True,
           title='Nuclear Share vs GDP per Capita (bubble = operating capacity)',
           labels={'gdp_per_capita':'GDP per capita (USD, log)', 'nuclear_share_percent':'Nuclear share (%)'},
           template='plotly_white').show()

## Capacity commissioned by decade

In [ ]:
r2 = reactors[reactors.commercial_operation_date.notna()].copy()
r2['decade'] = (r2.commercial_operation_date.dt.year // 10) * 10
by_dec = r2.groupby('decade')['capacity_mwe'].sum().div(1000).reset_index()
by_dec.columns = ['decade','capacity_gwe']

fig, ax = plt.subplots(figsize=(10,5))
sns.barplot(data=by_dec, x='decade', y='capacity_gwe', ax=ax, color='steelblue')
ax.set_title('Nuclear Capacity Commissioned per Decade (GWe)', fontsize=13)
ax.set_xlabel('Decade'); ax.set_ylabel('GWe')
plt.tight_layout(); plt.show()

**1970s–80s = peak buildout.** Post-TMI and Chernobyl the industry stalled. The recent uptick is driven primarily by China, UAE, and a handful of others.

## Missingness audit

In [ ]:
missing = reactors.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
fig, ax = plt.subplots(figsize=(10,4))
missing.plot.bar(ax=ax, color='salmon')
ax.set_title('Missing Data Fraction by Column')
ax.set_ylabel('Fraction missing')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()
print("Columns > 30% missing:"); print(missing[missing > 0.3].to_string())